# 16 · Linear channel equalization

A channel that is not flat in frequency (multipath, bandwidth limits,
residual dispersion) spreads each symbol into its neighbours: inter-symbol
interference.  If the channel response is known, a **linear equalizer**
undoes it in the frequency domain.  This notebook covers the fixed
(non-adaptive) case:

1. the ISI channel model;
2. **zero forcing** (ZF): invert the channel exactly;
3. **MMSE**: invert it as far as the noise allows;
4. measuring the channel: data-aided **channel sounding**, and equalizing
   with the measured response.

Adaptive equalizers, which learn the inverse from the data and track
changes, follow in [17](17-adaptive-equalization.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


tx = ck.generate(C16, num_symbols=2**15, symbol_rate=32e9, rng=1, device=DEVICE)

## 1. The ISI channel

At one sample per symbol, a linear channel is a causal FIR filter,

$$r[n] = \sum_k h[k]\, s[n-k] + w[n],$$

with frequency response `H(f) = Σ h[k] e^{-j2πfk}`.  Three channels of
increasing difficulty: a mild one, and two with a spectral notch of 20 and
30 dB, the kind of fade that two paths of nearly equal strength create.
Through the second path, each received symbol is the sum of the current
symbol and 0.9 times the previous one rotated by 90°: every point of the
grid splits into 16 copies, and the constellation is no longer readable.

In [ ]:
channels = {
    "mild": np.array([1, 0.3 + 0.2j, -0.1], dtype=np.complex64),
    "20 dB notch": np.array([1, 0.9j], dtype=np.complex64),
    "30 dB notch": np.array([1, 0.97j], dtype=np.complex64),
}
ESN0 = 18.0


def through(h):
    xp = tx.xp
    r = xp.convolve(tx.samples, xp.asarray(h))[: tx.samples.shape[-1]]
    # A fixed receiver noise floor: Es/N0 = 18 dB for the transmitted
    # unit-energy symbols, whatever the channel does to the received power.
    return ck.impairments.apply_awgn(
        tx.replace(samples=r), noise_var=10 ** (-ESN0 / 10), rng=2
    )


fig, (ax_f, ax_c) = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
f = np.fft.fftshift(np.fft.fftfreq(1024)) * tx.symbol_rate  # two-sided, in Hz
for name, h in channels.items():
    ck.plotting.plot_filter_response(
        h,
        sampling_rate=tx.symbol_rate,
        frequencies=f,
        quantity="magnitude",
        ax=ax_f,
        label=name,
        legend=True,
        title="Channel responses at one sample per symbol",
    )
ck.plotting.plot_constellation(
    through(channels["20 dB notch"]), ax=ax_c, title="Through the 20 dB notch"
)
plt.show()

## 2. Zero forcing

The zero-forcing equalizer applies `W(f) = 1 / H(f)`: the cascade is flat
and the ISI is gone.  `correct_channel(r, h)` does this blockwise in the
frequency domain (overlap-save), for SISO `(L,)` or MIMO `(C, C, L)`
impulse responses.  Its weakness is the noise: where `|H|` is small, `1/|H|`
is large, and the noise in that band is amplified with it.

On the mild channel the inverse removes the ISI and the noise grows only a
little.  On the 30 dB notch the same inverse multiplies the noise near the
notch by up to 30 dB: the ISI is gone, but the output is noisier than the
input.

In [ ]:
print(f"Es/N0 = {ESN0:.0f} dB")
for name, h in channels.items():
    r = through(h)
    zf = ck.equalization.correct_channel(r, h)
    print(
        f"{name:12s}: before {ck.metrics.snr(r):5.2f} dB, after ZF {ck.metrics.snr(zf):5.2f} dB"
    )

r = through(channels["mild"])
zf = ck.equalization.correct_channel(r, channels["mild"])
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(r, ax=axes[0], title="Mild channel, before")
ck.plotting.plot_constellation(zf, ax=axes[1], title="After zero forcing")
plt.show()

## 3. MMSE

The minimum-mean-square-error equalizer balances ISI against noise:

$$W(f) = \frac{H^*(f)}{|H(f)|^2 + \sigma^2},$$

which is `1/H` where the channel is strong and stays bounded in a notch,
accepting a little residual ISI there rather than amplifying the noise.
`correct_channel(..., noise_var=σ²)` switches to it; `σ²` is the noise
variance relative to the signal power, `10^(-Es/N0 / 10)`.

In [ ]:
sigma2 = 10 ** (-ESN0 / 10)
for name, h in channels.items():
    r = through(h)
    zf = ck.metrics.snr(ck.equalization.correct_channel(r, h))
    mmse = ck.metrics.snr(ck.equalization.correct_channel(r, h, noise_var=sigma2))
    print(f"{name:12s}: ZF {zf:5.2f} dB, MMSE {mmse:5.2f} dB")
ck.plotting.plot_channel_correction(
    channels["30 dB notch"],
    noise_var=sigma2,
    sampling_rate=tx.symbol_rate,
    title="30 dB notch: ZF against MMSE",
    show=True,
)

r = through(channels["30 dB notch"])
fig, axes = plt.subplots(
    1, 3, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 0.75))
)
for ax, (name, y) in zip(
    axes,
    (
        ("30 dB notch, before", r),
        ("ZF", ck.equalization.correct_channel(r, channels["30 dB notch"])),
        (
            "MMSE",
            ck.equalization.correct_channel(
                r, channels["30 dB notch"], noise_var=sigma2
            ),
        ),
    ),
    strict=True,
):
    ck.plotting.plot_constellation(
        y, ax=ax, title=f"{name}: {ck.metrics.snr(y):.1f} dB"
    )
plt.show()

## 4. Measuring the channel

The equalizers above were handed the true `h`.  A receiver measures it by
**channel sounding**: send known data `x`, receive `y`, and estimate
`H(f) = S_yx(f) / S_xx(f)` from averaged cross- and auto-spectra (the
Welch H1 estimator).  `estimate_channel(y, x)` returns a `ChannelEstimate`:
`H(f)` in FFT-bin order as `response`, and a truncated impulse response
centred on lag 0 as `taps`; for two polarizations it holds the full 2x2
response from one pass.

In [ ]:
nfft = 256
fig, ax = plt.subplots()
for name, h in channels.items():
    est = host(ck.equalization.estimate_channel(through(h), tx, nfft=nfft).response)
    frequencies = np.fft.fftfreq(nfft)
    true = ck.filtering.filter_response(h, frequencies=frequencies).response
    order = np.argsort(frequencies)
    line = ax.plot(
        np.fft.fftfreq(nfft)[order],
        20 * np.log10(np.abs(est[order])),
        label=f"{name}, estimated",
    )
    ax.plot(
        np.fft.fftfreq(nfft)[order],
        20 * np.log10(np.abs(true[order])),
        "--",
        color=line[0].get_color(),
    )
    print(f"{name:12s}: max |H_est - H| = {np.max(np.abs(est - true)):.3f}")
ax.set_xlabel("Frequency [symbol rates]")
ax.set_ylabel("|H(f)| [dB]")
ax.set_title("Channel sounding (dashed: true response)")
ax.legend()
plt.show()

### Sound, then equalize

With `num_taps=L` the estimate's taps have **lag 0 in the middle**, at
`center_tap = L // 2`: the taps before it are precursors, which a measured
channel can have (a sampling phase off the peak, an anticausal filter).
`correct_channel` takes the estimate and reads its centre tap; raw taps are
read with lag 0 at index 0 unless `center_tap=` says otherwise, so the same
taps without it come out `L // 2` symbols early.

In [ ]:
L = 31
for name, h in channels.items():
    r = through(h)
    est = ck.equalization.estimate_channel(r, tx, num_taps=L)
    known = ck.equalization.correct_channel(r, h, noise_var=sigma2)
    sounded = ck.equalization.correct_channel(r, est, noise_var=sigma2)
    print(
        f"{name:12s}: MMSE with true h {ck.metrics.snr(known):5.2f} dB, "
        f"with sounded h {ck.metrics.snr(sounded):5.2f} dB"
    )

**Why adaptive equalizers?**  A fixed equalizer needs the channel first
and assumes it stays put.  Real optical channels change: the polarization
rotates, PMD drifts, and the residual dispersion is unknown.  An adaptive
equalizer learns `W` from the data directly and keeps following it.

**Summary**

* ZF inverts `H(f)`; it removes all ISI but amplifies the noise in notches.
* MMSE adds `σ²` to the denominator and is never worse than ZF.
* `estimate_channel` sounds the channel from known data; the estimate goes
  straight into `correct_channel`, which reads its centre tap.

**Next:** [17 · Adaptive equalization](17-adaptive-equalization.ipynb).